# 👁️ TradeKriya "God's Eye" Agent Framework
### Autonomous Multi-Layer Market Intelligence & LinkedIn Editorial Synthesis Engine

The **God's Eye Agent** (*TradeKriya Argus*) is an omniscient quantitative agent that monitors all layers of the TradeKriya database:
1. **Macro Market Breadth**: Multi-timeframe moving average health (200 SMA, 50 SMA, 20 EMA), 10-day advance/decline flow, volume thrust accumulation, McClellan oscillator, and the **CMSI (Composite Market Strength Index 0–100)**.
2. **Quant Screener & S/R Zones**: Real-time classification of all 115 equities into **Breakouts (🚀)**, **Demand Rebounds (📈)**, **Supply Rejections (⚠️)**, and **Demand Breakdowns (🔻)**.
3. **5-Year Structural Personality DNA**: Evaluates 5-year breakout follow-through rates, demand bounce integrity, and 5Y Safety Scores.
4. **Domino Scenario Simulation**: Projects quantitative vulnerability if key support or resistance breaks.
5. **LinkedIn Editorial Engine**: Uses **Google Gemini 2.5 Flash** to synthesize this complex institutional telemetry into a high-engagement, viral LinkedIn market report.

In [26]:
import os
import sys
import json
import requests
import pandas as pd
import numpy as np
from pathlib import Path

# Setup Django Environment to access TradeKriya platform models and services
WORKSPACE_DIR = r"C:\Users\arunj\.gemini\antigravity\scratch\momentum_screener"
if WORKSPACE_DIR not in sys.path:
    sys.path.insert(0, WORKSPACE_DIR)

os.environ.setdefault("DJANGO_SETTINGS_MODULE", "momentum_screener.settings")

# Parse .env file for credentials
env_path = Path(WORKSPACE_DIR) / '.env'
if env_path.exists():
    with open(env_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith('#') and '=' in line:
                k, v = line.split('=', 1)
                clean_val = v.strip().strip("'").strip('"')
                os.environ[k.strip()] = clean_val
                os.environ[k.strip().upper()] = clean_val

import django
django.setup()

from screener.market_breadth_service import get_market_breadth_payload
from screener.quant_service import get_quant_terminal_payload, NpEncoder

GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY')
print("TradeKriya Platform Services Initialized!")
print("Gemini API Key Available:", bool(GEMINI_API_KEY))


TradeKriya Platform Services Initialized!
Gemini API Key Available: True


## Layer 1: Macro Market Breadth Telemetry
Extracts the latest market breadth metrics for both **Nifty F&O (115 broad market)** and **Nifty 50 (Bluechips)**.

In [27]:
# Ingest Market Breadth Payload
breadth_payload = get_market_breadth_payload(force_refresh=True)

breadth_all = breadth_payload['all']['summary']
breadth_n50 = breadth_payload['nifty50']['summary']

df_breadth_summary = pd.DataFrame([
    {
        "Universe": breadth_all['universe_name'],
        "Date": breadth_all['date_str'],
        "CMSI Score": f"{breadth_all['cmsi']} / 100",
        "Regime": breadth_all['regime_label'],
        "1M CMSI Shift": f"{breadth_all['cmsi_1m_shift']:+} pts",
        "% > 200 SMA": f"{breadth_all['pct_200']}% ({breadth_all['above_200_count']}/{breadth_all['total_stocks']})",
        "% > 50 SMA": f"{breadth_all['pct_50']}%",
        "% > 20 EMA": f"{breadth_all['pct_20']}%",
        "Daily Gainers %": f"{breadth_all['pct_gainers']}% ({breadth_all['gainers_count']} Adv / {breadth_all['losers_count']} Dec)",
        "10D Flow SMA": f"{breadth_all['gainers_10sma']}%",
        "Up-Volume %": f"{breadth_all['pct_up_vol']}%",
        "10D Volume Thrust": f"{breadth_all['vol_thrust_10sma']}%",
        "McClellan Osc": f"{breadth_all['mcclellan']:+}"
    },
    {
        "Universe": breadth_n50['universe_name'],
        "Date": breadth_n50['date_str'],
        "CMSI Score": f"{breadth_n50['cmsi']} / 100",
        "Regime": breadth_n50['regime_label'],
        "1M CMSI Shift": f"{breadth_n50['cmsi_1m_shift']:+} pts",
        "% > 200 SMA": f"{breadth_n50['pct_200']}% ({breadth_n50['above_200_count']}/{breadth_n50['total_stocks']})",
        "% > 50 SMA": f"{breadth_n50['pct_50']}%",
        "% > 20 EMA": f"{breadth_n50['pct_20']}%",
        "Daily Gainers %": f"{breadth_n50['pct_gainers']}% ({breadth_n50['gainers_count']} Adv / {breadth_n50['losers_count']} Dec)",
        "10D Flow SMA": f"{breadth_n50['gainers_10sma']}%",
        "Up-Volume %": f"{breadth_n50['pct_up_vol']}%",
        "10D Volume Thrust": f"{breadth_n50['vol_thrust_10sma']}%",
        "McClellan Osc": f"{breadth_n50['mcclellan']:+}"
    }
])

display(df_breadth_summary)


,Universe,Date,CMSI Score,Regime,1M CMSI Shift,% > 200 SMA,% > 50 SMA,% > 20 EMA,Daily Gainers %,10D Flow SMA,Up-Volume %,10D Volume Thrust,McClellan Osc
0,Nifty F&O (115 Equities),"Friday, 23 October 2026",26.7 / 100,Structural Distribution,-8.1 pts,19.6% (22/112),14.3%,16.1%,10.7% (12 Adv / 99 Dec),46.1%,14.0%,53.9%,-5.29
1,Nifty 50 (Bluechips),"Friday, 23 October 2026",27.5 / 100,Structural Distribution,-3.9 pts,14.6% (7/48),18.8%,18.8%,16.7% (8 Adv / 39 Dec),49.4%,25.3%,53.1%,-1.29


In [28]:
breadth_all

{'universe': 'all',
 'universe_name': 'Nifty F&O (115 Equities)',
 'date_str': 'Friday, 23 October 2026',
 'total_stocks': 112,
 'cmsi': 26.7,
 'cmsi_1m_shift': -8.1,
 'regime_label': 'Structural Distribution',
 'regime_status': 'STRUCTURAL_DISTRIBUTION',
 'regime_color': '#ef4444',
 'regime_desc': 'Widespread selling pressure. Stocks are shattering demand zones and slipping into falling cascade traps.',
 'playbook': 'Defensive capital preservation. Avoid dip buying. Tighten stop losses and increase cash allocation.',
 'above_200_count': 22,
 'pct_200': 19.6,
 'pct_200_1m_shift': -19.7,
 'pct_50': 14.3,
 'pct_50_1m_shift': -8.9,
 'pct_20': 16.1,
 'pct_20_1m_shift': -11.6,
 'gainers_count': 12,
 'losers_count': 99,
 'pct_gainers': 10.7,
 'gainers_10sma': 46.1,
 'gainers_10sma_1m_shift': 4.0,
 'pct_up_vol': 14.0,
 'vol_thrust_10sma': 53.9,
 'vol_thrust_1m_shift': 5.2,
 'mcclellan': -5.29,
 'mcclellan_1m_shift': -2.05,
 'net_52w': -18.8}

## Layer 2: Quant Screener Category Distribution & S/R Health
Scans all stocks in the Quant Terminal to evaluate where price is interacting with institutional demand and supply zones:
- **🚀 Breakout**: Stock clearing overhead supply ceiling with volume thrust.
- **📈 Rebound**: Stock bouncing off verified demand zone / accumulation floor.
- **⚠️ Rejection**: Stock failing at supply ceiling; seller dominance.
- **🔻 Breakdown**: Stock shattering demand floor; cascade risk.

In [29]:
records, charts_data = get_quant_terminal_payload()
df_stocks = pd.DataFrame(records)

# Category breakdown counts
cat_counts = df_stocks['category'].value_counts().to_dict()

breakouts = df_stocks[df_stocks['category'].str.contains('Breakout', case=False, na=False)]
rebounds = df_stocks[df_stocks['category'].str.contains('Rebound', case=False, na=False)]
rejections = df_stocks[df_stocks['category'].str.contains('Rejection', case=False, na=False)]
breakdowns = df_stocks[df_stocks['category'].str.contains('Breakdown', case=False, na=False)]
demand_zone = df_stocks[df_stocks['category'].str.contains('Demand Zone', case=False, na=False)]

print(f"=== QUANT TERMINAL S/R REGIME BREAKDOWN ({len(df_stocks)} Stocks) ===")
print(f"🚀 Breakouts:         {len(breakouts):2d} stocks ({len(breakouts)/len(df_stocks)*100:.1f}%)")
print(f"📈 Demand Rebounds:   {len(rebounds):2d} stocks ({len(rebounds)/len(df_stocks)*100:.1f}%)")
print(f"⚠️ Supply Rejections: {len(rejections):2d} stocks ({len(rejections)/len(df_stocks)*100:.1f}%)")
print(f"🔻 Demand Breakdowns: {len(breakdowns):2d} stocks ({len(breakdowns)/len(df_stocks)*100:.1f}%)")
print(f"🛡️ In Demand Zone:    {len(demand_zone):2d} stocks ({len(demand_zone)/len(df_stocks)*100:.1f}%)")

# Constructive vs Destructive Ratio
constructive = len(breakouts) + len(rebounds)
destructive = len(breakdowns) + len(rejections)
sr_health_ratio = round(constructive / max(1, destructive), 2)
print(f"\n⚖️ S/R Market Health Ratio (Constructive vs Destructive): {sr_health_ratio}x")

# Top representative candidates per bucket
top_breakouts = breakouts[['symbol', 'close', 'change_pct', 'vol_ratio', 'safety_score', 'supply_breakout_rate']].head(5).to_dict('records') if len(breakouts) else []
top_rebounds = rebounds[['symbol', 'close', 'change_pct', 'support_grade', 'support_score', 'safety_score', 'rr_ratio']].sort_values('rr_ratio', ascending=False).head(5).to_dict('records') if len(rebounds) else []
top_breakdowns = breakdowns[['symbol', 'close', 'change_pct', 'vol_ratio', 'safety_score']].head(5).to_dict('records') if len(breakdowns) else []
top_rejections = rejections[['symbol', 'close', 'change_pct', 'supply_rejection_rate', 'safety_score']].sort_values('supply_rejection_rate', ascending=False).head(5).to_dict('records') if len(rejections) else []


=== QUANT TERMINAL S/R REGIME BREAKDOWN (115 Stocks) ===
🚀 Breakouts:          9 stocks (7.8%)
📈 Demand Rebounds:    3 stocks (2.6%)
⚠️ Supply Rejections: 44 stocks (38.3%)
🔻 Demand Breakdowns:  0 stocks (0.0%)
🛡️ In Demand Zone:    46 stocks (40.0%)

⚖️ S/R Market Health Ratio (Constructive vs Destructive): 0.27x


In [32]:
demand_zone.struct_label.value_counts()

struct_label
Safe D&S Anchor               29
High Falling-Zone Risk         9
Mean-Reversion Oscillating     7
Expansion Runner               1
Name: count, dtype: int64

## Layer 3: Domino Scenario Engine ("What If Key Levels Break?")
Quantifies structural fragility by testing how many stocks are sitting within striking distance of their demand floors or supply ceilings.

In [33]:
# Convert numeric columns safely
df_stocks['demand_floor'] = pd.to_numeric(df_stocks['demand_floor'], errors='coerce')
df_stocks['supply_floor'] = pd.to_numeric(df_stocks['supply_floor'], errors='coerce')
df_stocks['close'] = pd.to_numeric(df_stocks['close'], errors='coerce')
df_stocks['safety_score'] = pd.to_numeric(df_stocks['safety_score'], errors='coerce')

# Vulnerability calculations:
df_stocks['dist_to_demand_pct'] = ((df_stocks['close'] - df_stocks['demand_floor']) / df_stocks['close'] * 100).round(2)
df_stocks['dist_to_supply_pct'] = ((df_stocks['supply_floor'] - df_stocks['close']) / df_stocks['close'] * 100).round(2)

# Stocks within 2.5% of breaking down below Demand Floor
vulnerable_to_breakdown = df_stocks[(df_stocks['dist_to_demand_pct'] >= -0.5) & (df_stocks['dist_to_demand_pct'] <= 2.5)]

# Stocks within 2.5% of breaking out above Supply Floor
primed_for_breakout = df_stocks[(df_stocks['dist_to_supply_pct'] >= -0.5) & (df_stocks['dist_to_supply_pct'] <= 2.5)]

# Falling Trap Risk: Low safety score (<50) stocks currently breaking down or rejected
high_risk_traps = df_stocks[(df_stocks['safety_score'] < 60) & (df_stocks['category'].str.contains('Breakdown|Rejection', case=False, na=False))]

print("=== KEY LEVEL SENSITIVITY & SCENARIO ANALYSIS ===")
print(f"⚠️ Stocks sitting on the edge of Demand Breakdown (within 2.5%): {len(vulnerable_to_breakdown)} stocks")
print(f"🚀 Stocks primed for Supply Breakout (within 2.5%):              {len(primed_for_breakout)} stocks")
print(f"☣️ High-Risk Falling Traps (Low Safety < 60 + Weak Zone):         {len(high_risk_traps)} stocks")

scenario_data = {
    "vulnerable_breakdown_count": len(vulnerable_to_breakdown),
    "vulnerable_breakdown_symbols": vulnerable_to_breakdown['symbol'].tolist()[:8],
    "primed_breakout_count": len(primed_for_breakout),
    "primed_breakout_symbols": primed_for_breakout['symbol'].tolist()[:8],
    "bear_scenario_inference": f"If Nifty breaks below immediate support, {len(vulnerable_to_breakdown)} stocks will trigger trailing stop-losses simultaneously, accelerating cascade selling into falling traps.",
    "bull_scenario_inference": f"If Nifty clears overhead resistance, {len(primed_for_breakout)} stocks have coiling energy to trigger a fresh multi-day breadth thrust with institutional volume backing."
}
print(f"\nBear Scenario: {scenario_data['bear_scenario_inference']}")
print(f"Bull Scenario: {scenario_data['bull_scenario_inference']}")


=== KEY LEVEL SENSITIVITY & SCENARIO ANALYSIS ===
⚠️ Stocks sitting on the edge of Demand Breakdown (within 2.5%): 88 stocks
🚀 Stocks primed for Supply Breakout (within 2.5%):              49 stocks
☣️ High-Risk Falling Traps (Low Safety < 60 + Weak Zone):         2 stocks

Bear Scenario: If Nifty breaks below immediate support, 88 stocks will trigger trailing stop-losses simultaneously, accelerating cascade selling into falling traps.
Bull Scenario: If Nifty clears overhead resistance, 49 stocks have coiling energy to trigger a fresh multi-day breadth thrust with institutional volume backing.


In [35]:
# high_risk_traps
primed_for_breakout

,symbol,close,change_pct,volume,vol_ratio,category,is_breakout,struct_label,struct_rec,safety_score,...,sma_45,sma_90,h52w,dist_52wh,l52w,dist_52wl,daily_vol,annual_vol,dist_to_demand_pct,dist_to_supply_pct
0,RELIANCE,1185.20,0.06,14213919,0.93,Consolidation / Equilibrium,False,Safe D&S Anchor,Safe for Demand-Buying (Institutional limit or...,80.5,...,1242.25,1274.06,1604.38,-26.13,1152.20,2.86,1.33,21.11,2.61,1.36
1,ADANIENT,2489.60,-2.37,1733102,1.10,Supply Rejection,False,High Falling-Zone Risk,AVOID Buying Dips (Cascade traps; demand level...,55.1,...,2881.23,2973.14,3245.00,-23.28,1753.00,42.02,2.31,36.70,4.41,-0.03
3,APOLLOHOSP,7705.50,1.33,486567,0.68,Demand Rebound,False,Safe D&S Anchor,Safe for Demand-Buying (Institutional limit or...,83.3,...,8468.76,8624.02,9070.50,-15.05,6696.50,15.07,1.66,26.35,0.91,1.33
5,AXISBANK,1212.00,0.00,7907742,1.01,In Demand Zone,False,Safe D&S Anchor,Safe for Demand-Buying (Institutional limit or...,92.0,...,1244.30,1268.12,1418.30,-14.55,1150.30,5.36,1.46,23.14,0.38,1.92
6,BAJAJ-AUTO,10074.00,-0.80,241763,0.78,Supply Rejection,False,Safe D&S Anchor,Safe for Demand-Buying (Institutional limit or...,85.0,...,11118.04,10919.89,12470.00,-19.21,8605.00,17.07,1.64,26.02,0.62,-0.06
7,BAJAJFINSV,1656.00,-1.31,944699,0.86,Supply Rejection,False,Safe D&S Anchor,Safe for Demand-Buying (Institutional limit or...,78.9,...,1848.43,1875.64,2195.00,-24.56,1597.00,3.69,1.55,24.65,1.53,0.87
8,BAJFINANCE,923.00,-2.03,6035288,0.69,Supply Rejection,False,Safe D&S Anchor,Safe for Demand-Buying (Institutional limit or...,83.2,...,1013.01,1028.39,1176.40,-21.54,787.90,17.15,2.05,32.61,2.92,0.42
12,CIPLA,1325.70,-0.39,1268569,1.30,In Demand Zone,False,Safe D&S Anchor,Safe for Demand-Buying (Institutional limit or...,88.9,...,1371.18,1404.43,1673.00,-20.76,1165.70,13.73,0.92,14.62,0.85,-0.47
13,COALINDIA,397.60,-1.23,4508328,0.76,In Demand Zone,False,Safe D&S Anchor,Safe for Demand-Buying (Institutional limit or...,88.6,...,414.29,420.90,491.25,-19.06,369.30,7.66,1.24,19.65,0.60,1.63
16,EICHERMOT,7020.00,-0.91,357910,0.86,In Demand Zone,False,Safe D&S Anchor,Safe for Demand-Buying (Institutional limit or...,92.4,...,7479.01,7568.34,8230.00,-14.70,6442.00,8.97,1.32,20.90,1.51,2.19


## Layer 4: God's Eye Omni-Telemetry Ingestion
Packages all layers of quantitative market data into a dense, unified JSON payload for LLM reasoning.

In [36]:
# Synthesize the complete God's Eye Telemetry Payload
gods_eye_payload = {
    "date": breadth_all['date_str'],
    "macro_breadth": {
        "nifty_fo_all": {
            "cmsi_score": breadth_all['cmsi'],
            "cmsi_regime": breadth_all['regime_label'],
            "cmsi_1m_shift": f"{breadth_all['cmsi_1m_shift']:+} pts",
            "pct_above_200_sma": f"{breadth_all['pct_200']}% ({breadth_all['above_200_count']}/{breadth_all['total_stocks']})",
            "pct_above_50_sma": f"{breadth_all['pct_50']}%",
            "pct_above_20_ema": f"{breadth_all['pct_20']}%",
            "daily_gainers_pct": f"{breadth_all['pct_gainers']}%",
            "adv_dec_ratio": f"{breadth_all['gainers_count']} Adv / {breadth_all['losers_count']} Dec",
            "gainers_10d_flow_sma": f"{breadth_all['gainers_10sma']}%",
            "volume_thrust_up_vol_pct": f"{breadth_all['pct_up_vol']}%",
            "vol_thrust_10d_sma": f"{breadth_all['vol_thrust_10sma']}%",
            "mcclellan_oscillator": f"{breadth_all['mcclellan']:+}"
        },
        "nifty_50_bluechips": {
            "cmsi_score": breadth_n50['cmsi'],
            "pct_above_200_sma": f"{breadth_n50['pct_200']}% ({breadth_n50['above_200_count']}/{breadth_n50['total_stocks']})",
            "daily_gainers_pct": f"{breadth_n50['pct_gainers']}%"
        }
    },
    "quant_screener_depth": {
        "total_universe": len(df_stocks),
        "breakouts": {
            "count": len(breakouts),
            "percentage": round(len(breakouts)/len(df_stocks)*100, 1),
            "notable_tickers": top_breakouts
        },
        "demand_rebounds": {
            "count": len(rebounds),
            "percentage": round(len(rebounds)/len(df_stocks)*100, 1),
            "notable_tickers": top_rebounds
        },
        "supply_rejections": {
            "count": len(rejections),
            "percentage": round(len(rejections)/len(df_stocks)*100, 1),
            "notable_tickers": top_rejections
        },
        "demand_breakdowns": {
            "count": len(breakdowns),
            "percentage": round(len(breakdowns)/len(df_stocks)*100, 1),
            "notable_tickers": top_breakdowns
        },
        "sr_health_ratio": sr_health_ratio
    },
    "key_level_scenario_analysis": scenario_data,
    "institutional_playbook": breadth_all['playbook']
}

print(json.dumps(gods_eye_payload, indent=2, cls=NpEncoder)[:800] + "...\n[Payload synthesized successfully]")


{
  "date": "Friday, 23 October 2026",
  "macro_breadth": {
    "nifty_fo_all": {
      "cmsi_score": 26.7,
      "cmsi_regime": "Structural Distribution",
      "cmsi_1m_shift": "-8.1 pts",
      "pct_above_200_sma": "19.6% (22/112)",
      "pct_above_50_sma": "14.3%",
      "pct_above_20_ema": "16.1%",
      "daily_gainers_pct": "10.7%",
      "adv_dec_ratio": "12 Adv / 99 Dec",
      "gainers_10d_flow_sma": "46.1%",
      "volume_thrust_up_vol_pct": "14.0%",
      "vol_thrust_10d_sma": "53.9%",
      "mcclellan_oscillator": "-5.29"
    },
    "nifty_50_bluechips": {
      "cmsi_score": 27.5,
      "pct_above_200_sma": "14.6% (7/48)",
      "daily_gainers_pct": "16.7%"
    }
  },
  "quant_screener_depth": {
    "total_universe": 115,
    "breakouts": {
      "count": 9,
      "percentage...
[Payload synthesized successfully]


## Layer 5: Autonomous AI Synthesis via Google Gemini 3.8 Flash
Generates a short, high-conviction, professional LinkedIn article with actionable trader takeaways.

In [37]:
SYSTEM_PROMPT = """You are TradeKriya's Senior Quantitative Market Strategist & Lead Portfolio Analyst.
Your task is to write a high-impact, professional, data-dense, yet accessible LinkedIn article summarizing today's Indian market action.

CRITICAL INSTRUCTIONS & EDITORIAL GUIDELINES:
1. Hook the reader immediately with the institutional ground reality vs. retail perception (e.g. index price vs CMSI market breadth).
2. Present exact quantitative telemetry from the provided JSON:
   - Composite Market Strength Index (CMSI) and regime.
   - Long-term structural health (% > 200 SMA) vs. short-term tactical flow (Gainers % & Up-Volume %).
   - Quant Terminal Screener Depth: Give the exact count and percentage of stocks in Breakout (🚀), Demand Rebound (📈), Supply Rejection (⚠️), and Demand Breakdown (🔻).
3. Include a dedicated section: '⚡ What Happens If Key Levels Break?' with the simulated Domino Scenario analysis (both Bearish breakdown cascade risk and Bullish breakout triggers).
4. Provide an Actionable Trading Playbook: Tell swing traders and positional investors exactly how to size capital, where to place stops, and which setups to favor tomorrow.
5. Highlight 2-3 specific stock setups surfaced by the screener with their S/R rationale (e.g. Demand Rebounds with high Reward-to-Risk or 5Y Safe Anchors).
6. Tone: Sharp, objective, institutional, engaging, formatted with clean bullet points, emojis, and professional spacing.
7. End with a thought-provoking discussion question and relevant hashtags (#TradeKriya #Nifty50 #MarketBreadth #QuantitativeTrading #StockMarketIndia).
"""

def generate_linkedin_article(telemetry_data, api_key):
    models = ["gemini-3.8-flash", "gemini-2.5-flash", "gemini-pro-latest"]
    
    prompt = f"""Here is the live TradeKriya God's Eye Telemetry Dump for today's market session ({telemetry_data['date']}):

```json
{json.dumps(telemetry_data, indent=2, cls=NpEncoder)}
```

Write the short, viral, institutional LinkedIn article now based strictly on these verified quantitative metrics.
"""

    payload = {
        "contents": [
            {
                "parts": [
                    {"text": SYSTEM_PROMPT + "\n\n" + prompt}
                ]
            }
        ],
        "generationConfig": {
            "temperature": 0.4,
            "maxOutputTokens": 4096
        }
    }

    last_err = None
    for model_name in models:
        url = f"https://generativelanguage.googleapis.com/v1beta/models/{model_name}:generateContent?key={api_key}"
        try:
            response = requests.post(url, json=payload, timeout=35)
            if response.status_code == 200:
                res_json = response.json()
                parts = res_json['candidates'][0]['content']['parts']
                text_content = "".join([p.get('text', '') for p in parts if 'text' in p])
                print(f"Generated successfully via {model_name} ({len(text_content)} chars)")
                return text_content
            else:
                last_err = f"{model_name} HTTP {response.status_code}: {response.text[:200]}"
                print(f"Model {model_name} returned {response.status_code}, trying next fallback...")
        except Exception as e:
            last_err = str(e)
            print(f"Model {model_name} exception: {e}, trying next fallback...")

    raise Exception(f"All Gemini models failed. Last error: {last_err}")

print("Synthesizing market telemetry via Google Gemini Engine...")
linkedin_article = generate_linkedin_article(gods_eye_payload, GEMINI_API_KEY)
print("\n" + "="*80)
print("             🚀 GENERATED LINKEDIN ARTICLE FROM GOD'S EYE AGENT")
print("="*80 + "\n")
print(linkedin_article)


Synthesizing market telemetry via Google Gemini Engine...
Model gemini-3.8-flash exception: HTTPSConnectionPool(host='generativelanguage.googleapis.com', port=443): Read timed out. (read timeout=35), trying next fallback...
Generated successfully via gemini-2.5-flash (5570 chars)

             🚀 GENERATED LINKEDIN ARTICLE FROM GOD'S EYE AGENT

---

### 🚨 **Indian Markets: The Illusion vs. The Underlying Reality – A Quantitative Deep Dive**

**Date:** Friday, 23 October 2026

While headline indices might paint a picture of resilience, TradeKriya's proprietary God's Eye Telemetry reveals a starkly different, and concerning, underlying reality for the Indian market today. As Senior Quantitative Market Strategist, my analysis points to a market grappling with significant structural weakness, demanding a highly defensive and disciplined approach from institutional and retail participants alike.

**The Hook: Index Perception vs. Institutional Ground Reality**

Don't be fooled by superficial 

## Layer 6: Visual Feed Simulation
Renders a simulated preview of how this post appears in a LinkedIn feed with an interactive executive summary card.

In [16]:
from IPython.display import display, HTML

# Escape HTML characters in article for safe display
import html
escaped_article = html.escape(linkedin_article)

html_preview = f"""
<div style="background: #0f172a; color: #f8fafc; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; border: 1px solid #334155; border-radius: 12px; padding: 24px; max-width: 740px; margin: 20px auto; box-shadow: 0 10px 25px rgba(0,0,0,0.5);">
    <!-- LinkedIn Author Header -->
    <div style="display: flex; align-items: center; gap: 12px; margin-bottom: 16px;">
        <div style="width: 48px; height: 48px; border-radius: 50%; background: linear-gradient(135deg, #e9805d, #38bdf8); display: flex; align-items: center; justify-content: center; font-weight: 800; font-size: 20px; color: #fff;">
            👁️
        </div>
        <div>
            <div style="font-weight: 700; font-size: 15px; color: #fff;">TradeKriya Quantitative Intelligence</div>
            <div style="font-size: 12px; color: #94a3b8;">Autonomous Market Breadth & Structural S/R Engine • Just now • 🌐</div>
        </div>
    </div>

    <!-- Quick Stats Grid -->
    <div style="display: grid; grid-template-columns: repeat(4, 1fr); gap: 10px; margin-bottom: 20px; background: #1e293b; padding: 12px; border-radius: 8px; border: 1px solid #334155;">
        <div style="text-align: center;">
            <div style="font-size: 10px; color: #94a3b8; text-transform: uppercase;">CMSI Score</div>
            <div style="font-size: 18px; font-weight: 800; color: #f59e0b;">{breadth_all['cmsi']} / 100</div>
            <div style="font-size: 10px; color: #34d399;">{breadth_all['cmsi_1m_shift']:+} pts 1M</div>
        </div>
        <div style="text-align: center;">
            <div style="font-size: 10px; color: #94a3b8; text-transform: uppercase;">&gt; 200 SMA</div>
            <div style="font-size: 18px; font-weight: 800; color: #38bdf8;">{breadth_all['pct_200']}%</div>
            <div style="font-size: 10px; color: #94a3b8;">{breadth_all['above_200_count']} Stocks</div>
        </div>
        <div style="text-align: center;">
            <div style="font-size: 10px; color: #94a3b8; text-transform: uppercase;">S/R Health</div>
            <div style="font-size: 18px; font-weight: 800; color: #10b981;">{sr_health_ratio}x</div>
            <div style="font-size: 10px; color: #94a3b8;">Const / Dest</div>
        </div>
        <div style="text-align: center;">
            <div style="font-size: 10px; color: #94a3b8; text-transform: uppercase;">Edge Alert</div>
            <div style="font-size: 18px; font-weight: 800; color: #f87171;">{len(vulnerable_to_breakdown)}</div>
            <div style="font-size: 10px; color: #f87171;">Near Breakdown</div>
        </div>
    </div>

    <!-- Article Body -->
    <div style="font-size: 13.5px; line-height: 1.65; color: #cbd5e1; white-space: pre-wrap; background: #131d2e; padding: 18px; border-radius: 8px; border: 1px solid #1e293b;">
{escaped_article}
    </div>

    <!-- Footer actions -->
    <div style="display: flex; justify-content: space-between; align-items: center; margin-top: 16px; border-top: 1px solid #334155; padding-top: 12px; font-size: 13px; color: #94a3b8;">
        <span>👍 142 Likes</span>
        <span>💬 28 Comments • 19 Reposts</span>
    </div>
</div>
"""

display(HTML(html_preview))
